# Quantum correlation corrections $K_SKK$
## Calculate the quantum correlation factors of $K_SKK$ background

### Include library for handling uncertainties
#### [Here is the ```uncertainties-cpp``` library on GitHub](https://github.com/Gattocrucco/uncertainties-cpp)

In [1]:
gInterpreter->AddIncludePath("/data/lhcb/users/tat/uncertainties-cpp");

In [2]:
#include<uncertainties/impl.hpp>
#include<uncertainties/ureal.hpp>
#include<uncertainties/io.hpp>
#include<uncertainties/math.hpp>
#include<uncertainties/stat.hpp>

### Load utility functions

In [3]:
gROOT->ProcessLine(".L ../UtilityFunctions.C");

### Number of bins

In [4]:
const int NumberBins = 4;

### Get fitted values of $c_i$, $s_i$ and $K_i$

In [5]:
const std::string cisiFilename("BiasCorrectedResults.txt");
auto cisiFitted = ParseParameters(cisiFilename);
std::map<int, uncertainties::udouble> ci, si, Ki;
for(int Bin = 1; Bin <= NumberBins; Bin++) {
    const std::string BinName = std::to_string(Bin);
    ci[Bin] = uncertainties::udouble(cisiFitted["c" + BinName], cisiFitted["c" + BinName + "_err"]);
    si[Bin] = uncertainties::udouble(cisiFitted["s" + BinName], cisiFitted["s" + BinName + "_err"]);
    Ki[Bin] = uncertainties::udouble(cisiFitted["K" + BinName], cisiFitted["K" + BinName + "_err"]);
    Ki[-Bin] = uncertainties::udouble(cisiFitted["Kbar" + BinName], cisiFitted["Kbar" + BinName + "_err"]);
}

### Get effective $F_+$ value for $K_SKK$

In [6]:
const auto FPlus_KSKK = GetFPlus("KSKK");

In [7]:
uncertainties::udouble FPlus_KSKK_unc(FPlus_KSKK.first, FPlus_KSKK.second);

### Calculate the quantum correlations factors in each bin for CP tags

In [8]:
const std::vector<std::string> Tags{
    "KK",
    "KSpi0",
    "pipipi0",
    "KKPartReco",
    "KSpi0PartReco",
    "pipipi0PartReco",
    "pipi",
    "KSpipipi0",
    "KSeta",
    "KSetaPrimepipieta",
    "KSetaPrimerhogamma",
    "KSpi0pi0",
    "KLpi0"
};
string QuantumCorrelationFactors;
for(const auto &Tag : Tags) {
    std::vector<uncertainties::udouble> QCFactors;
    for(int Bin = 1; Bin <= NumberBins; Bin++) {
        std::string Label = Tag + "_PeakingBackground0";
        Label += "_DoubleTag_CP_KKpipi_vs_" + Tag + "_SignalBin";
        Label += std::to_string(Bin) + "_QuantumCorrelationFactor";
        const auto FPlus_Tag = GetFPlus(Tag);
        uncertainties::udouble FPlus_Tag_unc(FPlus_Tag.first, FPlus_Tag.second);
        const auto KiFactor = uncertainties::sqrt(Ki[Bin]*Ki[-Bin])/(Ki[Bin] + Ki[-Bin]);
        const auto SigQCFactor = 1.0 - KiFactor*(2.0*FPlus_Tag_unc - 1.0)*ci[Bin];
        const auto BkgQCFactor = 1.0 - (2.0*FPlus_Tag_unc - 1.0)*(2.0*FPlus_KSKK_unc - 1.0);
        QCFactors.push_back(BkgQCFactor/SigQCFactor);
        QuantumCorrelationFactors += Label + " ";
        QuantumCorrelationFactors += std::to_string(uncertainties::nom(QCFactors.back())) + "\n";
        QuantumCorrelationFactors += Label + "_err ";
        QuantumCorrelationFactors += std::to_string(uncertainties::sdev(QCFactors.back())) + "\n";
    }
    QuantumCorrelationFactors += "\n";
    std::string Filename = "PeakingBackground_DT_KSKK_to_KKpipi_" + Tag;
    Filename += "_QuantumCorrelationFactors.root";
    std::vector<double> FlatCovMatrix =
        uncertainties::cov_matrix<std::vector<double>>(QCFactors);
    SaveCovMatrix(FlatCovMatrix, Filename);
}
//std::cout << QuantumCorrelationFactors;

### Save parameters to a file

In [9]:
std::ofstream File("QuantumCorrelationFactors_KSKK_pipipipi.txt");
File << QuantumCorrelationFactors;

### Function for loading $K_i$ for $K^0\pi\pi$

In [10]:
std::map<int, uncertainties::udouble> GetKi_K0pipi(const std::string &Tag) {
    std::map<std::string, double> FileDict;
    std::ifstream File("/data/bes3/tat/KKpipi_StrongPhase_Analysis/CommonInputs/Ki/" + Tag + "_Ki.txt");
    std::string Line;
    while(std::getline(File, Line)) {
        if(Line.empty()) {
            continue;
        }
        std::stringstream ss(Line);
        std::string Name;
        double Value;
        ss >> Name >> Value;
        FileDict.insert({Name, Value});
    }
    File.close();
    std::map<int, uncertainties::udouble> Ki;
    for(int i = 1; i <= 8; i++) {
        double Value = FileDict[Tag + "_K_m" + std::to_string(i)];
        double Error = FileDict[Tag + "_K_m" + std::to_string(i) + "_err"];
        Ki.insert({-i, uncertainties::udouble(Value, Error)});
        Value = FileDict[Tag + "_K_p" + std::to_string(i)];
        Error = FileDict[Tag + "_K_p" + std::to_string(i) + "_err"];
        Ki.insert({i, uncertainties::udouble(Value, Error)});
    }
    return Ki;
}

### Function for loading $c_i$ and $s_i$ for $K^0\pi\pi$

In [11]:
void Getcisi_K0pipi(const std::string &Tag,
                    std::map<int, uncertainties::udouble> &ci,
                    std::map<int, uncertainties::udouble> &si) {
    std::ifstream File("/data/bes3/tat/KKpipi_StrongPhase_Analysis/CommonInputs/cisi/" + Tag + "_cisi.txt");
    std::string Line;
    std::map<std::string, double> FileDict;
    while(std::getline(File, Line)) {
        if(Line.empty()) {
            continue;
        }
        std::stringstream ss(Line);
        std::string Name;
        double Value;
        ss >> Name >> Value;
        FileDict.insert({Name, Value});
    }
    File.close();
    for(int i = 1; i <= 8; i++) {
        double Value = FileDict[Tag + "_c" + std::to_string(i)];
        double Error = FileDict[Tag + "_c" + std::to_string(i) + "_err"];
        ci.insert({i, uncertainties::udouble(Value, Error)});
        Value = FileDict[Tag + "_s" + std::to_string(i)];
        Error = FileDict[Tag + "_s" + std::to_string(i) + "_err"];
        si.insert({i, uncertainties::udouble(Value, Error)});
    }
}

### Load $c_i$, $s_i$ and $K_i$ for $K^0\pi\pi$

In [12]:
auto Ki_KSpipi = GetKi_K0pipi("KSpipi");
std::map<int, uncertainties::udouble> ci_KSpipi, si_KSpipi;
Getcisi_K0pipi("KSpipi", ci_KSpipi, si_KSpipi);

In [13]:
auto Ki_KLpipi = GetKi_K0pipi("KLpipi");
std::map<int, uncertainties::udouble> ci_KLpipi, si_KLpipi;
Getcisi_K0pipi("KLpipi", ci_KLpipi, si_KLpipi);

### Calculate the quantum correlations factors in each bin for $K^0\pi\pi$ tags

In [14]:
const std::vector<std::string> Tags{
    "KSpipi",
    "KSpipiPartReco",
    "KLpipi"
};
std::string QuantumCorrelationFactors;
for(const auto &Tag : Tags) {
    const auto &Ki_Tag = (Tag == "KLpipi" ? Ki_KLpipi : Ki_KSpipi);
    const auto &ci_Tag = (Tag == "KLpipi" ? ci_KLpipi : ci_KSpipi);
    const auto &si_Tag = (Tag == "KLpipi" ? si_KLpipi : si_KSpipi);
    int KLpipiSign = (Tag == "KLpipi" ? -1 : +1);
    std::vector<uncertainties::udouble> QCFactors;
    for(int TagBin = 1; TagBin <= 8; TagBin++) {
        for(int Bin = -NumberBins; Bin <= NumberBins; Bin++) {
            if(Bin == 0) {
                continue;
            }
            std::string Label = Tag + "_PeakingBackground1";
            Label += "_DoubleTag_SCMB_KKpipi_vs_" + Tag + "_SignalBin";
            Label += (Bin > 0 ? "P" : "M") + std::to_string(TMath::Abs(Bin));
            Label += "_TagBin" + std::to_string(TagBin) + "_QuantumCorrelationFactor";
            const auto BkgKiFactor = KLpipiSign*uncertainties::sqrt(Ki_Tag.at(TagBin)*Ki_Tag.at(-TagBin))
                                    /(Ki_Tag.at(TagBin) + Ki_Tag.at(-TagBin));
            const auto BkgQCFactor = 1.0 - 2.0*BkgKiFactor*ci_Tag.at(TagBin)*(2.0*FPlus_KSKK_unc - 1.0);
            const auto SigKiFactor = uncertainties::sqrt(Ki[Bin]*Ki[-Bin]*Ki_Tag.at(TagBin)*Ki_Tag.at(-TagBin))
                                    /(Ki[Bin]*Ki_Tag.at(-TagBin) + Ki[-Bin]*Ki_Tag.at(TagBin));
            int siSign = (Bin > 0 ? +1 : -1);
            int AbsBin = TMath::Abs(Bin);
            const auto SigInterferenceTerm = ci[AbsBin]*ci_Tag.at(TagBin) + siSign*si[AbsBin]*si_Tag.at(TagBin);
            const auto SigQCFactor = 1.0 - 2.0*SigKiFactor*KLpipiSign*SigInterferenceTerm;
            QCFactors.push_back(BkgQCFactor/SigQCFactor);
            QuantumCorrelationFactors += Label + " ";
            QuantumCorrelationFactors += std::to_string(uncertainties::nom(QCFactors.back())) + "\n";
            QuantumCorrelationFactors += Label + "_err ";
            QuantumCorrelationFactors += std::to_string(uncertainties::sdev(QCFactors.back())) + "\n";
        }
    }
    QuantumCorrelationFactors += "\n";
    std::string Filename = "PeakingBackground_DT_KSKK_to_KKpipi_" + Tag;
    Filename += "_QuantumCorrelationFactors.root";
    std::vector<double> FlatCovMatrix =
        uncertainties::cov_matrix<std::vector<double>>(QCFactors);
    SaveCovMatrix(FlatCovMatrix, Filename);
}

In [15]:
File << QuantumCorrelationFactors;

### Calculate quantum correlation factors for the $\pi\pi\pi\pi$ background in $K_S\pi\pi$

In [16]:
const std::vector<std::string> Tags{
    "KSpipi",
    "KSpipiPartReco"
};
std::string QuantumCorrelationFactors;
const auto FPlus_4pi = GetFPlus("pipipipi");
uncertainties::udouble FPlus_4pi_unc(FPlus_4pi.first, FPlus_4pi.second);
for(const auto &Tag : Tags) {
    std::vector<uncertainties::udouble> QCFactors;
    for(int TagBin = 1; TagBin <= 8; TagBin++) {
        for(int Bin = -NumberBins; Bin <= NumberBins; Bin++) {
            if(Bin == 0) {
                continue;
            }
            std::string Label = Tag + "_PeakingBackground0";
            Label += "_DoubleTag_SCMB_KKpipi_vs_" + Tag + "_SignalBin";
            int AbsBin = TMath::Abs(Bin);
            Label += (Bin > 0 ? "P" : "M") + std::to_string(AbsBin);
            Label += "_TagBin" + std::to_string(TagBin) + "_QuantumCorrelationFactor";
            const auto BkgKiFactor = uncertainties::sqrt(Ki.at(Bin)*Ki.at(-Bin))/(Ki.at(Bin) + Ki.at(-Bin));
            const auto BkgQCFactor = 1.0 - 2.0*BkgKiFactor*ci.at(AbsBin)*(2.0*FPlus_4pi_unc - 1.0);
            const auto SigKiFactor = uncertainties::sqrt(Ki[Bin]*Ki[-Bin]*Ki_KSpipi.at(TagBin)*Ki_KSpipi.at(-TagBin))
                                    /(Ki[Bin]*Ki_KSpipi.at(-TagBin) + Ki[-Bin]*Ki_KSpipi.at(TagBin));
            int siSign = (Bin > 0 ? +1 : -1);
            const auto SigInterferenceTerm = ci[AbsBin]*ci_KSpipi.at(TagBin) + siSign*si[AbsBin]*si_KSpipi.at(TagBin);
            const auto SigQCFactor = 1.0 - 2.0*SigKiFactor*SigInterferenceTerm;
            QCFactors.push_back(BkgQCFactor/SigQCFactor);
            QuantumCorrelationFactors += Label + " ";
            QuantumCorrelationFactors += std::to_string(uncertainties::nom(QCFactors.back())) + "\n";
            QuantumCorrelationFactors += Label + "_err ";
            QuantumCorrelationFactors += std::to_string(uncertainties::sdev(QCFactors.back())) + "\n";
        }
    }
    QuantumCorrelationFactors += "\n";
    std::string Filename = "PeakingBackground_DT_4pi_to_" + Tag;
    Filename += "_QuantumCorrelationFactors.root";
    std::vector<double> FlatCovMatrix =
        uncertainties::cov_matrix<std::vector<double>>(QCFactors);
    SaveCovMatrix(FlatCovMatrix, Filename);
}

In [17]:
File << QuantumCorrelationFactors;

### Calculate quantum correlation factors for the $K_S\pi\pi$ background in $K_L\pi\pi$

In [18]:
std::string QuantumCorrelationFactors;
std::vector<uncertainties::udouble> QCFactors;
for(int TagBin = 1; TagBin <= 8; TagBin++) {
    for(int Bin = -NumberBins; Bin <= NumberBins; Bin++) {
        if(Bin == 0) {
            continue;
        }
        std::string Label = "KLpipi_PeakingBackground0";
        Label += "_DoubleTag_SCMB_KKpipi_vs_KLpipi_SignalBin";
        int AbsBin = TMath::Abs(Bin);
        Label += (Bin > 0 ? "P" : "M") + std::to_string(AbsBin);
        Label += "_TagBin" + std::to_string(TagBin) + "_QuantumCorrelationFactor";
        int siSign = (Bin > 0 ? +1 : -1);
        const auto BkgKiFactor = uncertainties::sqrt(Ki[Bin]*Ki[-Bin]*Ki_KSpipi.at(TagBin)*Ki_KSpipi.at(-TagBin))
                                /(Ki.at(Bin)*Ki_KSpipi.at(-TagBin) + Ki.at(-Bin)*Ki_KSpipi.at(TagBin));
        const auto BkgInterferenceTerm = ci[AbsBin]*ci_KSpipi.at(TagBin) + siSign*si[AbsBin]*si_KSpipi.at(TagBin);
        const auto BkgQCFactor = 1.0 - 2.0*BkgKiFactor*BkgInterferenceTerm;
        const auto SigKiFactor = uncertainties::sqrt(Ki[Bin]*Ki[-Bin]*Ki_KLpipi.at(TagBin)*Ki_KLpipi.at(-TagBin))
                                /(Ki[Bin]*Ki_KLpipi.at(-TagBin) + Ki[-Bin]*Ki_KLpipi.at(TagBin));
        const auto SigInterferenceTerm = ci[AbsBin]*ci_KLpipi.at(TagBin) + siSign*si[AbsBin]*si_KLpipi.at(TagBin);
        const auto SigQCFactor = 1.0 + 2.0*SigKiFactor*SigInterferenceTerm;
        QCFactors.push_back(BkgQCFactor/SigQCFactor);
        QuantumCorrelationFactors += Label + " ";
        QuantumCorrelationFactors += std::to_string(uncertainties::nom(QCFactors.back())) + "\n";
        QuantumCorrelationFactors += Label + "_err ";
        QuantumCorrelationFactors += std::to_string(uncertainties::sdev(QCFactors.back())) + "\n";
    }
}QuantumCorrelationFactors += "\n";
std::string Filename = "PeakingBackground_DT_KSpipi_to_KLpipi";
Filename += "_QuantumCorrelationFactors.root";
std::vector<double> FlatCovMatrix =
    uncertainties::cov_matrix<std::vector<double>>(QCFactors);
SaveCovMatrix(FlatCovMatrix, Filename);

In [19]:
File << QuantumCorrelationFactors;

In [20]:
File.close();